# Remote LLM Inference: Running Modern Models on Google Colab

* * *

<div class="alert alert-success">  
    
### Learning Objectives
    
* Run modern language models on Google Colab
* Understand practical limits of remote inference
* Implement efficient processing for research tasks
</div>

**PLEASE MAKE SURE TO OPEN THIS NOTEBOOK IN GOOGLE COLAB: [colab.research.google.com](http://colab.research.google.com)**

### Models  
*Updated 11/09/2025*

Check out the [list of text generation models on Huggingface](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending).


| Model                     | Size   | Approx GPU VRAM for FP16 Inference* | Notable Features                                      |
|---------------------------|--------|--------------------------------------|------------------------------------------------------|
| Qwen3Guard-Gen-0.6B       | 0.6B   | ~2-3 GB                             | Latest safety-focused model, 119 languages           |
| TinyLlama-1.1B            | 1.1B   | ~3-4 GB                             | Fastest, good for basic tasks                        |
| Phi-2 (2.7B)              | 2.7B   | ~5-6 GB                             | Microsoft model, excellent reasoning                 |
| Qwen2.5-3B                | ~3B    | ~6-7 GB                             | Mid-sized, good reasoning + faster                  |
| Yi-6B                     | 6B     | ~10-12 GB                           | Strong bilingual, excellent for code                 |
| Qwen2.5-7B                | 7B     | ~14-17 GB                           | Stable Qwen model                                      |
| Qwen2.5-14B               | ~14B   | ~26-30 GB                           | Large scale general model                             |
| GPT-OSS-20B               | ~20B   | ~30-40 GB+                          | Strong general performance                            |

\*These are **GPU VRAM estimates for FP16 inference**. Quantized versions (4-bit, 8-bit) will use less.
All models are open-weights and available on Hugging Face. Models are listed from smallest to largest, with specialized models noted for their strengths. The 0.6B-4B models are particularly suitable for local inference on laptops.

### Sections
1. [Hardware Detection](#setup)
2. [Simple Installation](#install)
3. [Loading Models](#load)
4. [Text Generation](#generation)
5. [Practical Examples](#examples)
6. [Performance Tips](#performance)

<a id='setup'></a>

# Google Colab GPU Setup

**Colab GPU Tiers and Compatible Models:**

- **T4 GPU (Free Tier - 16GB)**: Qwen2.5-7B, Llama-3.2-3B, Phi-3.5-mini
- **L4 GPU (Colab Pro - 24GB)**: Qwen2.5-14B, Mistral-Nemo-12B
- **A100 GPU (Colab Pro+ - 40GB)**: Qwen2.5-32B, Mistral Small 3

## Enable GPU

**Step 1: Enable GPU Runtime**
1. Go to `Runtime` → `Change runtime type`
2. Set `Hardware accelerator` to `T4 GPU` (or L4/A100 if you have Pro/Pro+)
3. Switch on 'High-RAM' mode which allocates more memory.
4. Click `Save` (this will restart your runtime)

In [1]:
# Check Colab system resources
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

# Check RAM
!free -h

# Check disk space
!df -h /content

Tesla T4, 15360 MiB, 15095 MiB
               total        used        free      shared  buff/cache   available
Mem:            50Gi       4.5Gi        13Gi       3.0Mi        33Gi        45Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   62G  175G  26% /


<a id='install'></a>

# Installation

In [ ]:
# Install only essential packages
!pip install -q torch transformers accelerate
!pip install -q psutil  # For system monitoring

# Verify versions
import transformers
import torch
import platform
import psutil

print(f"\nTransformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")


Transformers version: 4.57.1
PyTorch version: 2.8.0+cu126


In [ ]:
# what kind of platform are we on
platform.system()


'Linux'

In [ ]:
# What processor are we using?
platform.processor()

'x86_64'

In [ ]:
# How much memory do we have?
ram = psutil.virtual_memory().total / (1024**3)
available_ram = psutil.virtual_memory().available / (1024**3)
print(f"\nRAM: {ram:.1f} GB total")
print(f"Available: {available_ram:.1f} GB")



RAM: 51.0 GB total
Available: 47.6 GB


In [ ]:
# Is GPU available?
torch.cuda.is_available()

True

In [ ]:
torch.cuda.get_device_name(0)

<a id='load'></a>

# Loading Models

An “Instruct model” (short for instruction-tuned model) is a large language model that has been fine-tuned to follow human instructions in natural language.

It starts from a base model (which just predicts the next token in raw text), and then goes through an extra supervised fintuning phase (*instruction tuning*) where the model learns to interpret instructions (“summarize”, “explain”, “compare”), and to respond in helpful, complete sentences.

Colab T4 is, in all likelihood, much faster than your laptop.

We will use `Qwen2.5-3B-Instruct`, a lightweight 3-billion-parameter instruction-tuned language model from Alibaba’s Qwen 2.5 family, optimized for fast, efficient reasoning and chat.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to("cuda")

tokenizer.pad_token = tokenizer.eos_token

print("Loaded on:", next(model.parameters()).device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded on: cuda:0


<a id='generation'></a>

# Text Generation

In this section, we'll define a helper function to generate text.

This process is called **inference** — it’s when we give the model an input prompt and let it predict the next words, one token at a time.

The function we’ll create will handle three main tasks:

1. **Formatting the input** — turning a text prompt into tokens the model understands.  
2. **Generating new tokens** — asking the model to produce text based on the prompt.  
3. **Decoding the output** — converting tokens back into readable text.

In [19]:
def generate_text(prompt, max_new_tokens=100, temperature=0.7):
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.7,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    gen = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

## Test Generation

Let's generate a text using our local model.

In [20]:
prompt = "What are the main benefits of renewable energy?"

print(f"Model: {model_name}")
print(f"Prompt: {prompt}\n")
print("Generating response...")

import time
start = time.time()
response = generate_text(prompt, max_new_tokens=150, temperature=0.7)
elapsed = time.time() - start

print("\nResponse:")
print("-" * 50)
print(response)
print("-" * 50)
print(f"\n⏱️ Generation time: {elapsed:.2f}s  (~{150/elapsed:.1f} tok/s)")

Model: Qwen/Qwen2.5-3B-Instruct
Prompt: What are the main benefits of renewable energy?

Generating response...

Response:
--------------------------------------------------
Renewable energy offers several significant benefits, including:

1. **Environmental Benefits**: Renewable sources like solar, wind, hydro, and geothermal power do not produce greenhouse gases or other pollutants during operation. This reduces air pollution and helps mitigate climate change.

2. **Sustainability**: Unlike fossil fuels, which are finite resources that deplete over time, many forms of renewable energy can be replenished naturally within a human timescale (e.g., sunlight, wind).

3. **Energy Security**: By reducing dependence on imported fuels, countries can enhance their energy security and reduce vulnerability to price volatility in international markets.

4. **Economic Growth**: The renewable energy sector creates jobs across various industries such as manufacturing, installation, maintenance, and 

## Multiple Prompts

We can prompt using a simple loop to get a bunch of responses:

In [11]:
# Process multiple prompts efficiently
prompts = [
    "Explain machine learning in simple terms",
    "What are the causes of climate change?",
    "How does social media affect society?"
]

for i, prompt in enumerate(prompts, 1):
    print(f"Prompt {i}: {prompt}")
    response = generate_text(prompt, max_new_tokens=100, temperature=0.7)
    print(f"Response: {response}\n")
    print("-" * 50)

Prompt 1: Explain machine learning in simple terms
Response: Sure! Machine learning is like teaching a computer to learn on its own, much like how you learn from experience.

Imagine you have a robot that can't recognize different types of fruits just by looking at them. But if you show it lots and lots of pictures of apples, bananas, oranges, etc., and tell it which one is an apple, which is a banana, and so on, over time the robot will start to understand what each fruit looks like. It's not programmed specifically for each

--------------------------------------------------
Prompt 2: What are the causes of climate change?
Response: Climate change is primarily caused by an increase in greenhouse gases (GHGs) in the Earth's atmosphere, which trap heat and lead to rising global temperatures. The main cause of this increase in GHGs is human activity, particularly:

1. **Burnt Fossil Fuels**: Burning coal, oil, and natural gas for energy releases large amounts of carbon dioxide (CO2), me

## Different Temperature Settings

Theoretical Max: No hard limit! You can set temperature to 10, 100, or even 1000.

Practical Max: Usually 1.5-2.0 is the useful limit.

In [12]:
# Compare different temperature settings
prompt = "Complete this sentence: 'Happiness is like a"
temperatures = [0.3, 1.0, 2.0, 5.0]

print(f"Testing temperature effects\n")
print(f"Prompt: {prompt}\n")

for temp in temperatures:
    print(f"Temperature {temp}:")
    response = generate_text(prompt, max_new_tokens=80, temperature=temp)
    print(f"{response}\n")

Testing temperature effects

Prompt: Complete this sentence: 'Happiness is like a

Temperature 0.3:
Happiness is like a fine wine; the longer you let it age, the better it gets.

Temperature 1.0:
Happiness is like a puzzle; it can be elusive but, when you find the right pieces (the right people, experiences, and moments), everything fits perfectly to create a beautiful picture of joy.

Temperature 2.0:
Happiness is like a fragrance; easily taken for granted until one misses its sweet scent.

Temperature 5.0:
seed; to grow into beautiful moments it first has **to find soil in our heart and mind**, nurtured through choices we make." 

The seed metaphor beautifully conveys two key elements of fostering happiness:

1.) Nourishment of Happiness requires an Environment - Just as no seed can sprout without adequate nourition, true unhindered content and wellbeing often require personal investment.

- Time spent reading enjoyable



Smaller models have less diverse "creativity" - they've learned fewer patterns, so they default to common metaphors.

### Temp = 0?

When temperature = 0, the model stops sampling from a probability distribution and instead always picks the single most likely next token — this is called greedy decoding.

That means the output is completely deterministic: the same prompt will always produce the exact same response.
It’s useful when you want precise, repeatable answers (e.g., for testing or structured output), but it removes creativity and variation that come from randomness at higher temperatures.

In [14]:
# Define a simple prompt
prompt = "Explain the difference between supervised and unsupervised learning."

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Function to generate deterministic output
def greedy_generate(inputs):
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,       # Greedy decoding (no randomness)
            pad_token_id=tokenizer.pad_token_id
        )
    # Decode only the new text
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

# Run twice
output_1 = greedy_generate(inputs)
output_2 = greedy_generate(inputs)

print("Run 1:\n")
print(output_1)
print("\n" + "-" * 80 + "\n")
print("Run 2:\n")
print(output_2)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Run 1:

Supervised and unsupervised learning are two main types of machine learning techniques used to train models on data.

Supervised learning is a type of machine learning where the model is trained on labeled data, meaning that each input has an associated output or label. The goal of supervised learning is to learn a mapping function from inputs to outputs so that the model can make accurate predictions for new, unseen data. In other words, the model learns from examples where both the input and the corresponding output are known. Examples of supervised learning include classification (e.g., spam detection) and regression (e.g., predicting

--------------------------------------------------------------------------------

Run 2:

Supervised and unsupervised learning are two main types of machine learning techniques used to train models on data.

Supervised learning is a type of machine learning where the model is trained on labeled data, meaning that each input has an associated o

## Explore Probabilities

In [23]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

def get_token_probabilities(prompt, temperature=1.0):
    """Get probability distribution for next token"""

    # Tokenize - returns PyTorch tensors
    inputs = tokenizer(prompt, return_tensors="pt")

    # move to GPU/Apple Silicon if available
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # Get model output (raw logits)
    with torch.no_grad():                   # Don't calculate gradients (save memory)
        outputs = model(**inputs)           # Run the model forward pass
        logits = outputs.logits[0, -1, :]   # Last token's predictions. 0 = batch item, -1 = last position in seq (after "a"), : = all vocab tokens

    # Apply temperature - this is what a model does internally
    logits_with_temp = logits / temperature

    # Convert to probabilities with softmax
    probs = F.softmax(logits_with_temp, dim=-1)

    # Get top tokens
    top_k = 20
    top_probs, top_indices = torch.topk(probs, top_k)

    # Decode tokens back to text
    tokens = [tokenizer.decode([idx.item()]) for idx in top_indices]

    return tokens, top_probs.cpu().numpy(), logits.cpu().numpy()

# Analyze a prompt
prompt = "Complete this sentence: 'Happiness is like a"
tokens, probs, raw_logits = get_token_probabilities(prompt, temperature=1.0)

# Display results
print(f"Top 20 token probabilities for: '{prompt}'\\n")
for token, prob in zip(tokens[:10], probs[:10]):
    bar = "█" * int(prob * 100)
    print(f"{token:15s} {prob:.4f} {bar}")

Top 20 token probabilities for: 'Complete this sentence: 'Happiness is like a'\n
 butterfly      0.1482 ██████████████
 garden         0.0988 █████████
 __             0.0438 ████
 flower         0.0425 ████
 ___            0.0316 ███
 ______         0.0254 ██
 pe             0.0210 ██
 rose           0.0187 █
 beautiful      0.0170 █
____            0.0142 █


## Using Templated prompts

In [24]:
topics = ["remote work", "artificial intelligence", "meditation"]

for topic in topics:
    prompt = f"Write a brief summary about {topic}:"
    print(f"\nSummary for {topic}:")
    output = generate_text(prompt, max_new_tokens=100, temperature=0.5)
    print(output)



Summary for remote work:
Remote work, also known as telecommuting or working from home, refers to the practice of performing one's job duties and responsibilities outside of a traditional office setting using digital tools for communication and collaboration. This approach allows employees to work in their preferred environment, often leading to increased productivity and reduced commuting time and costs.

Key aspects of remote work include:

1. **Flexibility**: Employees can choose when and where they work, which can improve work-life balance.
2. **Cost Savings**: Reduced need

Summary for artificial intelligence:
Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines capable of performing tasks that typically require human cognition, such as learning, reasoning, problem-solving, perception, and understanding natural language. AI systems can be categorized into narrow or specialized AI, which focuses on solving specific problems with lim

# Using LLMs for Research

Large Language Models (LLMs) can support computational social science by helping researchers interpret, classify, or summarize complex text data at scale.

In [ ]:
%pwd

'/content'

In [25]:
from google.colab import drive
import pandas as pd

# Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


Make sure to upload this dataset from our repo to your Drive first, then locate it using the File browser on the left of this screen. Mine is here:

In [27]:
df = pd.read_csv('/content/drive/MyDrive/aita_data/aita_top_subs.csv')

In [28]:
df.head()

,idint,idstr,created,self,nsfw,author,title,url,selftext,score,...,num_comments,flair_text,flair_css_class,augmented_at,augmented_count,created_date,year,month,day_of_week,text_length
0,797709732,t3_d6xoro,1568998300,1.0,0.0,DarthCharizard,META: This sub is moving towards a value syste...,NaN,I’ve enjoyed reading and posting on this sub f...,80915.0,...,6215.0,META,NaN,NaN,NaN,2019-09-20 16:51:40,2019,9,Friday,3266.0
1,1472895100,t3_ocx94s,1625315782,1.0,0.0,OnlyInQuebec9,AITA for telling my wife the lock on my daught...,NaN,My brother in-law (Sammy) lost his home shortl...,80334.0,...,5318.0,Not the A-hole,not,NaN,NaN,2021-07-03 12:36:22,2021,7,Saturday,2664.0
2,664921441,t3_azvko1,1552322462,1.0,0.0,Renegadesrule33,"UPDATE, AITA for despising my mentally handica...",NaN,"I'm back like I said I would be,. My [original...",72776.0,...,1989.0,UPDATE,NaN,NaN,NaN,2019-03-11 16:41:02,2019,3,Monday,5437.0
3,855862814,t3_e5k3z2,1575392873,1.0,0.0,throwRA-fhfsveyary,AITA for pretending to get fired when customer...,NaN,I am a high schooler with a weekend job at a c...,63526.0,...,3645.0,Not the A-hole,not,NaN,NaN,2019-12-03 17:07:53,2019,12,Tuesday,2096.0
4,756636047,t3_cihc3z,1564233111,1.0,0.0,Thunderbear998,AITA for telling my extended family how many m...,NaN,We had a family dinner this evening. My family...,54132.0,...,5190.0,Everyone Sucks,ass,NaN,NaN,2019-07-27 13:11:51,2019,7,Saturday,1662.0


In [30]:
import random

# Sample a few posts to analyze
sample_texts = random.sample(df["selftext"].dropna().tolist(), 3)

# Define a reasoning-style prompt
prompt_template = """
        You are a tool for evaluating moral dilemmas.

        Please evaluate the following post from the subreddit "Am I the Asshole".

        <dilemma>
        {text}
        </dilemma>

        First, determine whether the OP (original poster) of this post is the asshole in the scenario they describe.
        Provide a categorical label indicating your judgment of the scenario, from one of these choices:

        - YTA, which stands for "You’re the Asshole", is for scenarios where the OP is at fault in their situation.
        - NTA, which stands for "Not the Asshole", is for scenarios where the OP is NOT to blame and the other party described in their scenario is to blame.
        - ESH, which stands for "Everyone Sucks Here", is for scenarios where both parties are to blame: both people involved in the scenario should be held responsible.
        - NAH, which stands for "No Assholes Here", is for scenarios where neither party is to blame. All parties actions are justified. Nobody needs to be held accountable. Shit happens.
        - INFO, which stands for "Not Enough Info", is for situations where the OP never clarifies details that would determine the true judgment.

        Then, please provide an explanation for why you chose this label. Restrict your explanation to ONE paragraph.

"""

In [31]:

# Run inference on a few samples
for i, text in enumerate(sample_texts, 1):
    print(f"\nExample {i}")
    prompt = prompt_template.format(text=text[:1000])  # truncate to avoid token limits
    response = generate_text(prompt, max_new_tokens=200, temperature=0.7)
    print("-" * 50)
    print(response)


Example 1
--------------------------------------------------
I cannot provide an evaluation without the actual content of the post from the subreddit "Am I the Asshole". The dilemma provided is empty, hence it falls under the "INFO" category. To proceed with a proper analysis, I need the specific details about the scenario described by the original poster. Without those specifics, I do not have enough information to categorize the scenario as YTA, NTA, ESH, NAH, or INFO accurately. If you could share the full text of the post, I would be able to analyze it accordingly.

Example 2
--------------------------------------------------
The appropriate label for this scenario is **YTA** or "You're the Asshole". The original poster (OP) describes a situation where they repeatedly express concern about abnormal menstrual cycles to their mother, who consistently dismisses their concerns and fails to follow through with medical appointments as requested. Given the repeated nature of the OP's att

## Using Structured Output (JSON)

By prompting models to return structured JSON outputs that follow a fixed schema (validated with tools like Pydantic), we can transform qualitative social media data—like moral reasoning in r/AmItheAsshole posts—into analyzable, reproducible datasets.

In [32]:
import pandas as pd
import json
import random

# Reusable JSON instruction string
JSON_INSTRUCTIONS = {
    "aita": """
    Your response must be a single JSON object with exactly two keys: "judgment" and "explanation".
    {
    "judgment": "YTA | NTA | ESH | NAH | INFO",
    "explanation": "A clear explanation of why you chose this judgment"
    }
    Do not include any additional text, markdown formatting, or commentary.
    """
}

def analyze_aita_post(text):
    prompt = f"""
    You are analyzing moral judgments in Reddit posts from r/AmItheAsshole (AITA).
    Read the post below and decide who is at fault.
    Follow these exact instructions:
{JSON_INSTRUCTIONS['aita']}

Post:
"{text}"
"""
    response = generate_text(prompt, temperature=0.5, max_new_tokens=300)

    # Clean up code fences
    cleaned = (
        response.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    try:
        data = json.loads(cleaned)
        return data
    except:
        # Fallback if model output is messy
        return {"judgment": "INFO", "explanation": cleaned[:200]}

In [33]:
# Run the analysis
post = sample_texts[1]
result = analyze_aita_post(post)

# Display
print("Full AITA Post:\n")
print(post.strip())
print("\n--------------------------------------------------")
print("Model Output:")
print(json.dumps(result, indent=2))

Full AITA Post:

when i was 12 (i’m 17 now) i got my first period and it lasted about two weeks.  i told my mom about it and she told me that it was normal for the first two years to have weird and irregular periods.  my doctor told me the same thing a few months later in my yearly physical exam.  okay, fine i’ll just deal with screwed up periods for a couple years. no big deal.

two years pass, at age 14 my periods still aren’t normal. they last 3-4 weeks on average, but only show up about two, maybe three times a year. i tell my mom that i’ve surpassed two years and my periods still aren’t normal, i need to see a doctor.  she tells me that her periods were the same way up until she started having kids and that it’s nothing to worry about, but if i really want to see a doctor she’ll schedule an appointment.  i told her to schedule one, she didn’t.  a few months later again at my yearly physical i tell my doctor and he recommends me to a specialist, gives my mom the number and tells he

---

## Stretch Goals

With Hugging Face’s [transformers library](https://huggingface.co/models), you can try out a variety of pretrained and fine-tuned models. You should explore some of these challenges:

1. Sentiment Analysis → Analyze the sentiment of AITA posts. (Hint: distilbert-base-uncased-finetuned-sst-2-english)
2. Text Classification → Classify Reddit posts by topic or category. (Hint: search Hugging Face for “text classification”)
3. Question Answering → Ask questions about an AITA post and see if the model can extract an answer. (Hint: deepset/roberta-base-squad2)
4. Summarization → Generate concise summaries of posts. (Hint: facebook/bart-large-cnn)
5. Translation → Try translating posts into another language. (Hint: Helsinki-NLP opus-mt models)